# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [1]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [2]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # 1) make sure autograd tracks x. If the caller's tensor is not tracked, work on a
    #    detached copy so we never flip requires_grad on somebody else's tensor.
    if not x.requires_grad:
        x = x.detach().requires_grad_(True)
    # 2) forward pass builds the graph  x -> x**2 -> sum -> f   (f is a scalar)
    f = (x ** 2).sum()
    # 3) ask autograd for df/dx directly. It returns a tuple (one entry per input)
    #    and does NOT write anything into x.grad.
    (dfdx,) = torch.autograd.grad(f, x)
    return dfdx

# ---- f(x) = sum(x^2)  =>  df/dx = 2x ----
x = torch.tensor([1.0, -2.0, 3.0])            # plain tensor, requires_grad=False
print("x             ", x.tolist(), " requires_grad =", x.requires_grad)
print("df/dx         ", grad_with_autograd_grad(x).tolist())
print("expected 2*x  ", (2 * x).tolist())
print()
print("x.grad after the call:", x.grad, " <- autograd.grad RETURNS the gradient, it never touches .grad")
print("calling twice gives the same answer:",
      grad_with_autograd_grad(x).tolist(), grad_with_autograd_grad(x).tolist())
print()
# works just as well for a tensor that already requires grad, and for any shape
x2 = torch.arange(6, dtype=torch.float32).reshape(2, 3).requires_grad_(True)
print("2x3 input -> gradient has the same shape:", tuple(grad_with_autograd_grad(x2).shape))

x              [1.0, -2.0, 3.0]  requires_grad = False
df/dx          [2.0, -4.0, 6.0]
expected 2*x   [2.0, -4.0, 6.0]

x.grad after the call: None  <- autograd.grad RETURNS the gradient, it never touches .grad
calling twice gives the same answer: [2.0, -4.0, 6.0] [2.0, -4.0, 6.0]

2x3 input -> gradient has the same shape: (2, 3)


In [3]:
def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    # .backward() only fills .grad of LEAF tensors, and it ADDS into .grad.
    # So we always work on a fresh leaf: a detached copy that shares x's data but
    # starts with grad == None. That guarantees
    #   (a) it is a leaf, so .grad gets populated,
    #   (b) nothing from a previous call can accumulate into it,
    #   (c) the caller's own x.grad is left untouched.
    leaf = x.detach().requires_grad_(True)
    f = (leaf ** 2).sum()
    f.backward()                      # writes df/dleaf into leaf.grad
    return leaf.grad

# ---- same function, this time through .backward() and .grad ----
x = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
print("df/dx         ", grad_with_backward(x).tolist())
print("expected 2*x  ", (2 * x.detach()).tolist())
print("second call   ", grad_with_backward(x).tolist(), " <- still 2x, nothing accumulated")
print("caller's x.grad is still", x.grad)
print()

# ---- the pitfall this function avoids: backward() ADDS into .grad ----
w = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
(w ** 2).sum().backward()
print("after 1st backward      w.grad =", w.grad.tolist())
(w ** 2).sum().backward()
print("after 2nd backward      w.grad =", w.grad.tolist(), " <- 2x + 2x: accumulated!")
w.grad = None                                   # or w.grad.zero_()  /  optimizer.zero_grad()
(w ** 2).sum().backward()
print("after reset + backward  w.grad =", w.grad.tolist(), " <- resetting between calls fixes it")

df/dx          [2.0, -4.0, 6.0]
expected 2*x   [2.0, -4.0, 6.0]
second call    [2.0, -4.0, 6.0]  <- still 2x, nothing accumulated
caller's x.grad is still None

after 1st backward      w.grad = [2.0, -4.0, 6.0]
after 2nd backward      w.grad = [4.0, -8.0, 12.0]  <- 2x + 2x: accumulated!
after reset + backward  w.grad = [2.0, -4.0, 6.0]  <- resetting between calls fixes it


In [4]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # make sure BOTH inputs are tracked (detached copies, so the caller's tensors are untouched)
    if not a.requires_grad:
        a = a.detach().requires_grad_(True)
    if not b.requires_grad:
        b = b.detach().requires_grad_(True)
    f = (a ** 2 + a * b).sum()
    # pass a tuple of inputs -> get a tuple of gradients back, in the same order
    dfda, dfdb = torch.autograd.grad(f, (a, b))
    return dfda, dfdb

# ---- f(a, b) = sum(a^2 + a*b)  =>  df/da = 2a + b,   df/db = a ----
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
dfda, dfdb = grad_wrt_multiple_inputs(a, b)
print("df/da           ", dfda.tolist())
print("expected 2a + b ", (2 * a + b).tolist())
print("df/db           ", dfdb.tolist())
print("expected a      ", a.tolist())
print()

# ---- each gradient has the shape of ITS OWN input, even when the forward pass broadcasts ----
a = torch.ones(2, 3)                 # (2, 3)
b = torch.tensor([1.0, 2.0, 3.0])    # (3,)  broadcast against every row of a
dfda, dfdb = grad_wrt_multiple_inputs(a, b)
print("a", tuple(a.shape), "-> df/da", tuple(dfda.shape), dfda.tolist())
print("b", tuple(b.shape), "-> df/db", tuple(dfdb.shape), dfdb.tolist(), " <- = a.sum(0): b was used in both rows")

df/da            [12.0, 24.0, 36.0]
expected 2a + b  [12.0, 24.0, 36.0]
df/db            [1.0, 2.0, 3.0]
expected a       [1.0, 2.0, 3.0]

a (2, 3) -> df/da (2, 3) [[3.0, 4.0, 5.0], [3.0, 4.0, 5.0]]
b (3,) -> df/db (3,) [2.0, 2.0, 2.0]  <- = a.sum(0): b was used in both rows


## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [5]:
from torch.utils.data import DataLoader, Dataset

In [6]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        assert x.shape[0] == y.shape[0], f"x has {x.shape[0]} samples but y has {y.shape[0]}"
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]

# ---- wrap (x, y) pairs; DataLoader will call __len__ and __getitem__ for us ----
x = torch.arange(10.0).reshape(5, 2)          # 5 samples of 2 features
y = torch.tensor([0, 1, 0, 1, 1])
ds = TensorPairDataset(x, y)
print("len(ds) =", len(ds))
xi, yi = ds[2]
print("ds[2]   =", xi.tolist(), yi.item())
print("x[2]    =", x[2].tolist(), y[2].item(), " <- same thing, just indexed")

len(ds) = 5
ds[2]   = [4.0, 5.0] 0
x[2]    = [4.0, 5.0] 0  <- same thing, just indexed


In [7]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        assert tokens.dim() == 2, "tokens must be (N, T)"
        self.tokens = tokens

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        seq = self.tokens[idx]          # (T,), integer dtype is preserved by slicing
        return seq[:-1], seq[1:]        # input_ids, target_ids: each (T-1,)

# ---- shift by one: the model sees token t and must predict token t+1 ----
tokens = torch.tensor([[10, 11, 12, 13, 14],
                       [20, 21, 22, 23, 24]])
ds = NextTokenDataset(tokens)
inp, tgt = ds[0]
print("row 0      ", tokens[0].tolist())
print("input_ids  ", inp.tolist(), " shape", tuple(inp.shape), inp.dtype)
print("target_ids ", tgt.tolist(), " <- input shifted left by one")
print("len(ds) =", len(ds), " (one item per row)")

row 0       [10, 11, 12, 13, 14]
input_ids   [10, 11, 12, 13]  shape (4,) torch.int64
target_ids  [11, 12, 13, 14]  <- input shifted left by one
len(ds) = 2  (one item per row)


In [8]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        assert 0 < crop_len <= tokens.shape[1], "crop_len must fit inside T_total"
        self.tokens = tokens
        self.crop_len = crop_len
        # seed given  -> private generator: crops are reproducible and independent of the global RNG
        # seed absent -> generator=None, i.e. torch's global RNG
        self.generator = torch.Generator().manual_seed(seed) if seed is not None else None

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        max_start = self.tokens.shape[1] - self.crop_len          # inclusive upper bound for s
        s = int(torch.randint(0, max_start + 1, (1,), generator=self.generator))
        return self.tokens[idx, s : s + self.crop_len]

# ---- random fixed-length windows; a seed makes the sequence of windows reproducible ----
tokens = torch.arange(30).reshape(3, 10)      # row r holds 10r .. 10r+9
ds_a = RandomCropSequenceDataset(tokens, crop_len=4, seed=0)
ds_b = RandomCropSequenceDataset(tokens, crop_len=4, seed=0)
print("ds_a[1] ->", ds_a[1].tolist())
print("ds_b[1] ->", ds_b[1].tolist(), " <- same seed, same crop")
print("ds_a[1] ->", ds_a[1].tolist(), " <- next draw from the generator, usually a different window")
print("every crop has shape", tuple(ds_a[2].shape), "and stays inside its own row:", ds_a[2].tolist())

ds_a[1] -> [14, 15, 16, 17]
ds_b[1] -> [14, 15, 16, 17]  <- same seed, same crop
ds_a[1] -> [13, 14, 15, 16]  <- next draw from the generator, usually a different window
every crop has shape (4,) and stays inside its own row: [22, 23, 24, 25]


In [9]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    lengths = torch.tensor([int(s.numel()) for s in seqs], dtype=torch.long)
    # pad_sequence stacks along a new batch dim and fills the tail of shorter rows with pad_value
    tokens = nn.utils.rnn.pad_sequence([s.to(torch.long) for s in seqs], batch_first=True, padding_value=pad_value)
    lengths = lengths.to(tokens.device)
    # position t is padding iff it lies at or beyond the row's real length: broadcast (1,T_max) >= (B,1)
    padding_mask = torch.arange(tokens.shape[1], device=tokens.device)[None, :] >= lengths[:, None]
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)

# ---- three ragged rows -> one rectangular (B, T_max) block plus a mask ----
seqs = [torch.tensor([5, 6, 7]), torch.tensor([8]), torch.tensor([9, 10], dtype=torch.int32)]
pb = pad_1d_sequences(seqs, pad_value=-1)
print("tokens       ", pb.tokens.tolist(), pb.tokens.dtype, " <- int32 row was cast to long")
print("lengths      ", pb.lengths.tolist())
print("padding_mask ", pb.padding_mask.tolist(), " <- True exactly where t >= length")

tokens        [[5, 6, 7], [8, -1, -1], [9, 10, -1]] torch.int64  <- int32 row was cast to long
lengths       [3, 1, 2]
padding_mask  [[False, False, False], [False, True, True], [False, False, True]]  <- True exactly where t >= length


In [10]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    B = len(batch)
    # pad inputs and targets in ONE call so both get the same T_max and the same padding layout
    padded = pad_1d_sequences([x for x, _ in batch] + [y for _, y in batch], pad_value=pad_value)
    padding_mask = padded.padding_mask[:B]
    return {
        "input_ids": padded.tokens[:B],
        "target_ids": padded.tokens[B:],
        "attention_mask": ~padding_mask,
        "padding_mask": padding_mask,
    }

# ---- variable-length (input, target) pairs -> one padded dict ----
items = [(torch.tensor([1, 2, 3]), torch.tensor([2, 3, 4])),
         (torch.tensor([7]),       torch.tensor([8]))]
out = collate_next_token_batch(items)
for k, v in out.items():
    print(f"{k:15s}", tuple(v.shape), str(v.dtype).replace("torch.", ""), v.tolist())
print("attention_mask is ~padding_mask:", torch.equal(out["attention_mask"], ~out["padding_mask"]))

input_ids       (2, 3) int64 [[1, 2, 3], [7, 0, 0]]
target_ids      (2, 3) int64 [[2, 3, 4], [8, 0, 0]]
attention_mask  (2, 3) bool [[True, True, True], [True, False, False]]
padding_mask    (2, 3) bool [[False, False, False], [False, True, True]]
attention_mask is ~padding_mask: True


In [11]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,      # None -> DataLoader's default stacking
        num_workers=num_workers,
    )

# ---- default collate: stacks fixed-size items along a new batch dim ----
ds = TensorPairDataset(torch.arange(10.0).reshape(5, 2), torch.arange(5))
dl = make_dataloader(ds, batch_size=2, shuffle=False, drop_last=True)
for xb, yb in dl:
    print("x batch", tuple(xb.shape), " y batch", yb.tolist())
print("5 samples / batch 2 with drop_last=True ->", len(dl), "batches (the odd one is dropped)")
print()
# ---- custom collate: ragged next-token items need our padding function ----
# a plain Python list already has __len__ and __getitem__, so it works as a map-style dataset
ragged = [(torch.tensor([1, 2, 3]), torch.tensor([2, 3, 4])),
          (torch.tensor([7]),       torch.tensor([8])),
          (torch.tensor([4, 5]),    torch.tensor([5, 6]))]
dl = make_dataloader(ragged, batch_size=2, shuffle=False, collate_fn=collate_next_token_batch)
for b in dl:
    print({k: tuple(v.shape) for k, v in b.items()})

x batch (2, 2)  y batch [0, 1]
x batch (2, 2)  y batch [2, 3]
5 samples / batch 2 with drop_last=True -> 2 batches (the odd one is dropped)

{'input_ids': (2, 3), 'target_ids': (2, 3), 'attention_mask': (2, 3), 'padding_mask': (2, 3)}
{'input_ids': (1, 2), 'target_ids': (1, 2), 'attention_mask': (1, 2), 'padding_mask': (1, 2)}


## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [12]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
        m = torch.zeros_like(p)     # same shape / dtype / device as p, requires_grad=False
        v = torch.zeros_like(p)
    return AdamWState(m=m, v=v, t=0)

# ---- state mirrors the parameter: same shape / dtype / device, zeros, step 0 ----
p = nn.Parameter(torch.randn(2, 3))
st = init_adamw_state(p)
print("m", tuple(st.m.shape), st.m.dtype, st.m.device, " all zero:", bool((st.m == 0).all()))
print("v", tuple(st.v.shape), " requires_grad:", st.v.requires_grad, " <- state is never part of the graph")
print("t =", st.t)

m (2, 3) torch.float32 cpu  all zero: True
v (2, 3)  requires_grad: False  <- state is never part of the graph
t = 0


In [13]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas
    state.t += 1                                   # bias correction uses the NEW step count
    t = state.t
    with torch.no_grad():                          # p is usually a leaf that requires grad
        state.m.mul_(beta1).add_(grad, alpha=1 - beta1)            # m = b1*m + (1-b1)*g
        state.v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)  # v = b2*v + (1-b2)*g^2
        m_hat = state.m / (1 - beta1 ** t)
        v_hat = state.v / (1 - beta2 ** t)
        # decoupled weight decay: wd*p is added to the UPDATE, not to the gradient
        p.sub_(lr * (m_hat / (v_hat.sqrt() + eps) + weight_decay * p))
    return state

# ---- sanity check against torch.optim.AdamW: same param, same grads, same hyper-params ----
torch.manual_seed(0)
p_torch = torch.randn(3, 4, dtype=torch.float64, requires_grad=True)
p_mine = p_torch.detach().clone()
state = init_adamw_state(p_mine)
opt = torch.optim.AdamW([p_torch], lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
for _ in range(5):
    g = torch.randn_like(p_torch)
    g_copy = g.clone()
    p_torch.grad = g.clone()
    opt.step()
    state = adamw_step_(p_mine, g, state, lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
    print(f"step {state.t}: max |mine - torch| = {(p_mine - p_torch.detach()).abs().max().item():.1e}")
print("grad left untouched by the step:", torch.equal(g, g_copy))

step 1: max |mine - torch| = 4.4e-16
step 2: max |mine - torch| = 2.2e-16
step 3: max |mine - torch| = 2.2e-16
step 4: max |mine - torch| = 2.2e-16
step 5: max |mine - torch| = 2.2e-16
grad left untouched by the step: True


In [14]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    assert len(params) == len(grads) == len(states), "params, grads and states must line up"
    return [
        adamw_step_(p, g, s, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        for p, g, s in zip(params, grads, states)
    ]

# ---- a whole layer's worth of parameters, checked against torch again ----
torch.manual_seed(1)
ref = nn.Linear(4, 2).double()
params = [p.detach().clone() for p in ref.parameters()]
states = [init_adamw_state(p) for p in params]
opt = torch.optim.AdamW(ref.parameters(), lr=1e-2, weight_decay=0.1)
for _ in range(3):
    grads = [torch.randn_like(p) for p in params]
    for p, g in zip(ref.parameters(), grads):
        p.grad = g.clone()
    opt.step()
    states = adamw_step_many_(params, grads, states, lr=1e-2, weight_decay=0.1)
print("steps taken per param:", [s.t for s in states])
print("max |mine - torch| per param:",
      [f"{(p - r.detach()).abs().max().item():.1e}" for p, r in zip(params, ref.parameters())])

steps taken per param: [3, 3]
max |mine - torch| per param: ['2.8e-17', '6.9e-18']


## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [15]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    x, y = batch
    model.train()                            # 1) train mode (dropout on, batchnorm updates stats)
    optimizer.zero_grad()                    # 2) reset .grad, otherwise backward() accumulates
    pred = model(x)                          # 3) forward
    loss = nn.functional.mse_loss(pred, y)   # 4) loss
    loss.backward()                          # 5) backward: fills .grad of every parameter
    optimizer.step()                         # 6) update parameters
    return loss.item()                       # python float, detached from the graph

# ---- fit y = x @ w + b with plain SGD; the loss should fall over the steps ----
torch.manual_seed(0)
model = nn.Linear(3, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.randn(64, 3)
y = x @ torch.tensor([[1.0], [-2.0], [0.5]]) + 0.3
losses = [train_step_mse(model, (x, y), opt) for _ in range(60)]
print("loss every 10 steps:", [round(l, 4) for l in losses[::10]])
print("return type:", type(losses[0]).__name__,
      " learned w:", [round(v, 2) for v in model.weight.detach().flatten().tolist()],
      " b:", round(model.bias.item(), 2), " (true: [1, -2, 0.5], 0.3)")

loss every 10 steps: [5.7509, 0.2616, 0.0154, 0.001, 0.0001, 0.0]
return type: float  learned w: [1.0, -2.0, 0.5]  b: 0.3  (true: [1, -2, 0.5], 0.3)


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [16]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    assert weight.dim() >= 2, "fan in/out need at least 2 dims"
    receptive_field = weight.shape[2:].numel()     # 1 for Linear, kH*kW for a conv kernel
    fan_in = weight.shape[1] * receptive_field
    fan_out = weight.shape[0] * receptive_field
    return fan_in, fan_out

# ---- Linear weight (out, in) and conv kernel (out, in, kH, kW), checked against torch ----
lin = nn.Linear(5, 3).weight            # (3, 5)
conv = torch.empty(8, 4, 3, 3)          # (out=8, in=4, 3x3 kernel)
for name, w in [("Linear (3,5)", lin), ("Conv (8,4,3,3)", conv)]:
    print(f"{name:15s} ours {fan_in_fan_out(w)}   torch {tuple(nn.init._calculate_fan_in_and_fan_out(w))}")

Linear (3,5)    ours (5, 3)   torch (5, 3)
Conv (8,4,3,3)  ours (36, 72)   torch (36, 72)


In [17]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * (6.0 / (fan_in + fan_out)) ** 0.5
    with torch.no_grad():                    # weight may be an nn.Parameter: never track the init
        weight.uniform_(-bound, bound)
    return weight

# ---- U(-bound, bound) with bound = sqrt(6 / (fan_in + fan_out)) ----
w = xavier_uniform_(torch.empty(200, 300))
fan_in, fan_out = fan_in_fan_out(w)
bound = (6 / (fan_in + fan_out)) ** 0.5
print(f"bound {bound:.4f}   max|w| {w.abs().max().item():.4f}   (never exceeds the bound)")
print(f"std   ours {w.std().item():.4f}   theory {bound / 3 ** 0.5:.4f}   "
      f"torch {nn.init.xavier_uniform_(torch.empty(200, 300)).std().item():.4f}")
print("returns the same tensor object, initialised in place:", xavier_uniform_(w) is w)

bound 0.1095   max|w| 0.1095   (never exceeds the bound)
std   ours 0.0631   theory 0.0632   torch 0.0634
returns the same tensor object, initialised in place: True


In [18]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    fan_in, _ = fan_in_fan_out(weight)
    gain = nn.init.calculate_gain(nonlinearity)    # sqrt(2) for "relu"
    std = gain / fan_in ** 0.5
    bound = 3 ** 0.5 * std                          # U(-b, b) has std b / sqrt(3)
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight

# ---- U(-bound, bound) with bound = sqrt(3) * sqrt(2) / sqrt(fan_in) = sqrt(6 / fan_in) ----
w = kaiming_uniform_(torch.empty(200, 300))
fan_in, _ = fan_in_fan_out(w)
bound = (6 / fan_in) ** 0.5
print(f"bound {bound:.4f}   max|w| {w.abs().max().item():.4f}")
print(f"std   ours {w.std().item():.4f}   theory {(2 / fan_in) ** 0.5:.4f}   "
      f"torch {nn.init.kaiming_uniform_(torch.empty(200, 300), nonlinearity='relu').std().item():.4f}")
print("only fan_in matters here, so a (200, 300) and a (50, 300) weight get the same bound:",
      f"{kaiming_uniform_(torch.empty(50, 300)).abs().max().item():.4f}")

bound 0.1414   max|w| 0.1414
std   ours 0.0817   theory 0.0816   torch 0.0816
only fan_in matters here, so a (200, 300) and a (50, 300) weight get the same bound: 0.1414


In [19]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    if scheme == "xavier":
        xavier_uniform_(layer.weight)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight, nonlinearity="relu")
    elif scheme == "zero":
        with torch.no_grad():
            layer.weight.zero_()
    else:
        raise ValueError(f"unknown init scheme: {scheme!r}")
    if layer.bias is not None:
        with torch.no_grad():
            layer.bias.zero_()               # bias is always zeroed
    return layer

# ---- the three schemes on the same layer shape; the bias is always zeroed ----
for scheme in ["xavier", "kaiming_relu", "zero"]:
    layer = init_linear_(nn.Linear(300, 200), scheme=scheme)
    print(f"{scheme:13s} weight std {layer.weight.std().item():.4f}   bias all zero: {bool((layer.bias == 0).all())}"
          f"   still an nn.Parameter: {isinstance(layer.weight, nn.Parameter)}")
try:
    init_linear_(nn.Linear(2, 2), scheme="glorot")
except ValueError as e:
    print("unknown scheme ->", e)

xavier        weight std 0.0633   bias all zero: True   still an nn.Parameter: True
kaiming_relu  weight std 0.0814   bias all zero: True   still an nn.Parameter: True
zero          weight std 0.0000   bias all zero: True   still an nn.Parameter: True
unknown scheme -> unknown init scheme: 'glorot'
